In [ ]:
# -*- coding: utf-8 -*-
"""
api_benchmark.py
------------------
阶段二：通用 API 接入 + 多 API 性能评测工具。

和阶段一的区别：
    阶段一只能固定调用某一个写死的 API（DeepSeek）；
    阶段二不写死任何具体 API，接入信息（API Key / Base URL / 模型名）
    全部由使用者在"配置区"填写或运行时输入，只要目标服务遵循
    OpenAI 的接口协议（messages / chat.completions），无论是 DeepSeek、
    其他厂商、还是自建的兼容服务，都可以直接接进来对比测试。

本文件按 Jupyter Notebook 使用习惯，划分成几个"区块"（用分隔线标注），
你可以整份复制到一个 cell 里跑，也可以按分隔线拆成多个 cell 分别跑
（拆分时注意：区块之间有依赖关系，必须按从上到下的顺序依次执行）。

依赖：
    pip install openai
"""

import csv
import os
import time
import datetime
from typing import List, Dict, Tuple, Optional

from openai import OpenAI, APIConnectionError, APIStatusError, APITimeoutError


# ============================================================
# 区块一：通用 API 客户端（不绑定任何具体厂商）
# ============================================================
class ApiClient:
    """
    通用 API 客户端：只要是遵循 OpenAI 接口协议的服务都能用这个类调用。

    和"写死 DeepSeek"版本的唯一区别是：这里额外保存了一个 name 字段，
    用来在实验结果里标记"这条记录是哪个 API 跑出来的"，
    方便阶段二做多个 API 之间的横向对比。
    """

    def __init__(
    self,
    name: str,
    api_key: str,
    base_url: str,
    model: str,
    temperature: float = 0.7,
    max_tokens: int = 1000,
    stream: bool = False,
    timeout: float = 180.0
    ):

        self.name = name
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.stream = stream

        self.client = OpenAI(
        api_key=api_key,
        base_url=base_url,
        timeout=timeout
        )

    def chat(self, prompt: str) -> Dict:
        """
        发送一条 Prompt，返回 (回复文本, 耗时秒数)。

        每次实验都是独立请求，不携带历史对话，方便做"单轮输入 -> 单轮输出"
        的干净对比。
        """
        messages = [{"role": "user", "content": prompt}]
        start_time = time.time()

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=self.temperature,
                max_tokens=self.max_tokens,
                stream=self.stream,
                stream_options={
                "include_usage": True
                } if self.stream else None
            )
            if self.stream:
                chunks = []
                first_token_delay = None
                total_tokens = None
                
                for chunk in response:
                    print("DEBUG:", chunk)

                    # 第一个有效返回
                    if first_token_delay is None:
                        first_token_delay = time.time() - start_time
                    # 获取文本
                    if chunk.choices:
                        if chunk.choices[0].delta.content:
                            chunks.append(
                                chunk.choices[0].delta.content
                            )
                    # 获取token
                    if chunk.usage:
                        total_tokens = chunk.usage.total_tokens


                reply = "".join(chunks)

                elapsed = time.time() - start_time


                return {
                    "reply": reply,
                    "elapsed": elapsed,
                    "first_token_delay": first_token_delay,
                    "tokens": total_tokens,
                    "raw_json": {},
                    "status_code": 200,
                    "error": ""
                }
            else:
                reply = response.choices[0].message.content
                elapsed = time.time() - start_time
                if response.usage:
                    total_tokens = response.usage.total_tokens
                else:
                    total_tokens = None


                return {
                    "reply": reply,
                    "elapsed": elapsed,
                    "first_token_delay": None,
                    "tokens": total_tokens,
                    "raw_json": response.model_dump(),
                    "status_code": 200,
                    "error": ""
                }


        except Exception as e:
            return {
                "reply":"",
                "elapsed":None,
                "first_token_delay":None,
                "tokens":None,
                "raw_json":"",
                "status_code":None,
                "error":str(e)
            }
        except APIConnectionError as e:
            raise RuntimeError(f"[{self.name}] 无法连接到 API 服务：{e}") from e
        except APITimeoutError as e:
            raise RuntimeError(f"[{self.name}] 请求超时，请稍后重试：{e}") from e
        except APIStatusError as e:
            raise RuntimeError(
                f"[{self.name}] API 返回错误状态码 {e.status_code}：{e.message}"
            ) from e


In [ ]:

test = clients[0].chat("你好，介绍一下人工智能")

print(test)


In [ ]:
#print(os.getcwd())

In [ ]:
# ============================================================
# 区块二：在这里配置你想对比测试的 API 列表
# ============================================================
# 这是唯一需要你手动填写的地方。想测试几个 API，就在列表里加几项。
# api_key 可以留空字符串 ""，运行时程序会提示你手动输入
# （这样密钥就不会明文写在代码里，更安全）。
#
# 示例：下面预置了一个 DeepSeek 的配置，你可以照着格式再加一个别的 API：

    # 再加一个要对比的 API，取消注释并按实际情况修改：
    # {
    #     "name": "示例API2",
    #     "api_key": "",
    #     "base_url": "https://example.com/v1",
    #     "model": "example-model",
    # },
API_PROFILES_CONFIG = []
def input_api_config():

    print("请输入 API 配置信息")

    name = input("API名称：")

    api_key = input("API Key：")

    base_url = input(
        "Base URL："
    )

    model = input(
        "模型名称："
    )

    temperature = float(
        input("temperature（默认0.7）：") or 0.7
    )

    max_tokens = int(
        input("max_tokens（默认1000）：") or 1000
    )

    stream_input = input(
        "是否开启流式输出(True/False)："
    )

    stream = stream_input.lower()=="true"


    return {
        "name":name,
        "api_key":api_key,
        "base_url":base_url,
        "model":model,
        "temperature":temperature,
        "max_tokens":max_tokens,
        "stream":stream
    }

In [ ]:
# ============================================================
# 区块三：CSV 结果记录
# ============================================================
RESULT_CSV_PATH = "result.csv"
CSV_HEADERS = ["测试方式",
    "时间戳",
    "模型",
    "Prompt",
    "全链路耗时",
    "首字节延迟",
    "返回内容",
    "token消耗",
    "HTTP状态码",
    "原始JSON",
    "异常信息"]


def append_result_to_csv(data):

    file_exists=os.path.isfile(
        RESULT_CSV_PATH
    )

    with open(
        RESULT_CSV_PATH,
        "a",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        writer=csv.writer(f)

        if not file_exists:
            writer.writerow(CSV_HEADERS)


        writer.writerow([
            data["method"],
            data["timestamp"],
            data["model"],
            data["prompt"],
            data["elapsed"],
            data["first_token_delay"],
            data["reply"],
            data["tokens"],
            data["status_code"],
            data["raw_json"],
            data["error"]
        ])


In [ ]:
# ============================================================
# 区块四：根据配置构建客户端（缺密钥时会提示输入）
# ============================================================
def build_clients():

    config = input_api_config()

    client = ApiClient(
        name=config["name"],
        api_key=config["api_key"],
        base_url=config["base_url"],
        model=config["model"],
        temperature=config["temperature"],
        max_tokens=config["max_tokens"],
        stream=config["stream"]
    )

    return [client]

In [ ]:
def load_prompts(file):

    prompts=[]

    with open(
        file,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:
            if line.strip():
                prompts.append(line.strip())

    return prompts

In [ ]:
from typing import List
def load_prompts(file_path):
    prompts=[]

    with open(file_path,"r",encoding="utf-8") as f:
        for line in f:
            if line.strip():
                prompts.append(line.strip())

    return prompts
# ============================================================
# 区块五：主流程 —— 交互式输入 Prompt，对所有已配置的 API 各跑一次
# ============================================================
def run_experiment(clients: List[ApiClient]) -> None:
    """
    循环提示用户输入 Prompt；每输入一次，会对 clients 里的每一个
    API 配置都调用一次、各自计时、各自打印结果，并各自写入 result.csv
    （用"API名称"这一列区分是哪个 API 跑出来的），方便直接在表格里
    按 Prompt 分组，横向比较不同 API 的表现。
    """
    if not clients:
        print("没有可用的 API 客户端，请检查 API_PROFILES_CONFIG 配置。")
        return

    print("-" * 60)
    print(f"本次实验将对比以下 {len(clients)} 个 API：")
    for c in clients:
        print(f"  - {c.name}（模型：{c.model}）")
    print(f"实验结果将保存到：{os.path.abspath(RESULT_CSV_PATH)}")
    print("正在读取 prompts.txt，开始批量测试...")
    print("-" * 60)

    # 读取批量测试Prompt
    prompts = load_prompts("testprompt.txt")


    for user_input in prompts:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # 同一个Prompt发送给所有API
        for client in clients:
            print(f"\n>>> 正在调用【{client.name}】...")

            try:
                result = client.chat(user_input)
                reply = result["reply"]
                elapsed = result["elapsed"]
                total_tokens = result["tokens"]
                status_code = result["status_code"]
                raw_json = result["raw_json"]
                error = result["error"]
                first_token_delay = result["first_token_delay"]

            except RuntimeError as e:
                print(f"[错误] {e}")
                continue


            print(f"Prompt：{user_input}")
            print(f"回答：{reply}")
            print(f"异常信息：{result['error']}")
            if elapsed is not None:
                print(f"耗时：{elapsed:.2f} 秒")
            else:
                print("耗时：None")
            
            print(f"Token：{total_tokens}")
            if result["first_token_delay"] is not None:
                print(
                    f"首字节延迟：{result['first_token_delay']:.4f} 秒"
                )
            else:
                print("首字节延迟：None")


            data = {
                "method": "API",
                "timestamp": timestamp,
                "model": client.name,
                "prompt": user_input,
                "elapsed": elapsed,
                "first_token_delay": result["first_token_delay"],
                "reply": reply,
                "tokens": total_tokens,
                "status_code": result["status_code"],
                "raw_json": str(result["raw_json"]),
                "error": error
            }
            append_result_to_csv(data)
            
            print(f"\n所有Prompt测试完成，结果保存到 {RESULT_CSV_PATH}")

In [ ]:

def load_prompts(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    # 用 === 分割多个不同的 Prompt
    prompts = [p.strip() for p in content.split("===") if p.strip()]
    return prompts
prompts = load_prompts("testprompt.txt")

print(prompts)

print(len(prompts))


In [ ]:
# ============================================================
# 区块六：运行入口
# ============================================================
# 在 Jupyter Notebook 里，把上面几个区块依次跑完之后，
# 最后单独用一个 cell 执行下面这两行即可开始实验：
#
#     clients = build_clients(API_PROFILES_CONFIG)
#     run_experiment(clients)
#
# 如果是当作普通 .py 脚本用 `python api_benchmark.py` 运行，
# 下面的 if __name__ == "__main__" 会自动帮你执行。
if __name__ == "__main__":
    clients = build_clients()
    run_experiment(clients)

In [ ]:
'''
import inspect

print(inspect.getsource(run_experiment))
import os

print(os.getcwd())

print(os.listdir())
'''